# Chapter 18: RAG — Retrieval-Augmented Generation

LLMs have a fundamental limitation: their knowledge is **frozen at training time**.

```
User: What was Apple's revenue last quarter?
LLM:  I don't have access to recent financial data. (trained months ago)
```

**RAG** solves this by giving the LLM access to external documents at query time:

```
User question → Search relevant docs → Stuff docs into prompt → LLM answers
```

The LLM doesn't need to memorize everything — it just needs to read the right documents.

---
## RAG vs Fine-Tuning: When to Use Which

| | RAG | Fine-Tuning |
|---|---|---|
| **Best for** | Facts that change | Behavior/style that's stable |
| **Data freshness** | Always up-to-date | Frozen at training time |
| **Cost** | Storage + retrieval | GPU training |
| **Hallucination** | Lower (has source docs) | Higher (from memory) |
| **Example** | Company knowledge base | Medical writing style |

RAG = **what** the model knows (facts).
Fine-tuning = **how** the model behaves (style/format).

In [ ]:
import numpy as np
np.random.seed(42)

# RAG Pipeline Overview
print("The RAG Pipeline:\n")
print("  ┌──────────────────────────────────────────────────────────────┐")
print("  │                    INDEXING (done once)                       │")
print("  │                                                              │")
print("  │  Documents → Chunk → Embed → Store in Vector DB             │")
print("  │                                                              │")
print("  └──────────────────────────────────────────────────────────────┘")
print()
print("  ┌──────────────────────────────────────────────────────────────┐")
print("  │                   QUERYING (every request)                    │")
print("  │                                                              │")
print("  │  User Query → Embed → Search Vector DB → Top-K docs         │")
print("  │       ↓                                        ↓             │")
print("  │  ┌─────────────────────────────────────────────────┐        │")
print("  │  │ Prompt = 'Given these documents: {docs}         │        │")
print("  │  │          Answer this question: {query}'         │        │")
print("  │  └─────────────────────────────────────────────────┘        │")
print("  │       ↓                                                      │")
print("  │  LLM generates answer (grounded in retrieved docs)           │")
print("  │                                                              │")
print("  └──────────────────────────────────────────────────────────────┘")

---
## Step 1: Chunking — Splitting Documents

Documents are too long to fit in the LLM's context. We split them into **chunks**.

```
10-page document → 20 chunks of ~500 tokens each
```

Chunking strategies:
- **Fixed size**: every N tokens (simple but might split mid-sentence)
- **Sentence-based**: split at sentence boundaries
- **Semantic**: split at topic changes
- **Overlap**: chunks overlap by N tokens (prevents losing context at boundaries)

In [ ]:
# Chunking strategies
print("Document Chunking Strategies:\n")

document = (
    "Python is a high-level programming language. It was created by Guido van Rossum. "
    "Python emphasizes code readability. It uses significant indentation. "
    "Python supports multiple paradigms. These include procedural, object-oriented, "
    "and functional programming. Python has a large standard library. "
    "It is often described as a batteries-included language. "
    "Python is used in web development, data science, AI, and automation."
)

print(f"  Document ({len(document)} chars): '{document[:60]}...'\n")

# Strategy 1: Fixed-size chunks
def chunk_fixed(text, chunk_size=80):
    return [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]

fixed_chunks = chunk_fixed(document, 100)
print(f"  1. Fixed-size (100 chars):")
for i, chunk in enumerate(fixed_chunks):
    print(f"     Chunk {i+1}: '{chunk}'")

# Strategy 2: Sentence-based
def chunk_sentences(text, max_sentences=2):
    sentences = [s.strip() + '.' for s in text.split('.') if s.strip()]
    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunks.append(' '.join(sentences[i:i+max_sentences]))
    return chunks

sent_chunks = chunk_sentences(document, 2)
print(f"\n  2. Sentence-based (2 sentences per chunk):")
for i, chunk in enumerate(sent_chunks):
    print(f"     Chunk {i+1}: '{chunk[:70]}{'...' if len(chunk) > 70 else ''}'")

# Strategy 3: With overlap
def chunk_overlap(text, chunk_size=100, overlap=30):
    chunks = []
    for i in range(0, len(text), chunk_size - overlap):
        chunk = text[i:i+chunk_size]
        if chunk.strip():
            chunks.append(chunk)
    return chunks

overlap_chunks = chunk_overlap(document, 120, 30)
print(f"\n  3. With overlap (120 chars, 30 overlap):")
for i, chunk in enumerate(overlap_chunks[:3]):
    print(f"     Chunk {i+1}: '{chunk[:60]}...'")
print(f"     (overlap ensures no info lost at boundaries)")

---
## Step 2: Embedding — Converting Chunks to Vectors

To search documents by meaning (not just keywords), we convert each chunk to a **vector** (Ch 11).

```
"Python is used in AI" → [0.8, -0.2, 0.5, ...] (768+ dimensions)
"Machine learning uses Python" → [0.7, -0.1, 0.6, ...] (similar vector!)
"The weather is nice today" → [-0.3, 0.9, -0.1, ...] (very different)
```

Similar meanings → nearby vectors → we can find relevant chunks by vector distance.

In [ ]:
# Simulate embedding and vector search
print("Embedding Chunks for Semantic Search:\n")

# Simulated document chunks (a knowledge base)
knowledge_base = [
    "Python was created by Guido van Rossum in 1991.",
    "Python uses indentation instead of braces for code blocks.",
    "TensorFlow and PyTorch are popular deep learning frameworks.",
    "RAG combines retrieval with language model generation.",
    "The Python package manager is called pip.",
    "Neural networks are inspired by biological neurons.",
    "Transformers use self-attention mechanisms.",
    "Python 3.12 added support for type parameter syntax.",
]

# Simulate embeddings (in reality, use sentence-transformers or OpenAI API)
# We'll create embeddings that cluster semantically similar docs
embedding_dim = 8

# Hand-craft embeddings to show semantic similarity
embeddings = np.array([
    [0.9, 0.1, 0.2, 0.0, 0.8, 0.1, 0.0, 0.1],  # Python history
    [0.8, 0.2, 0.1, 0.0, 0.7, 0.3, 0.0, 0.2],  # Python syntax
    [0.2, 0.9, 0.8, 0.1, 0.1, 0.1, 0.7, 0.3],  # ML frameworks
    [0.1, 0.7, 0.3, 0.9, 0.0, 0.2, 0.5, 0.8],  # RAG
    [0.7, 0.1, 0.1, 0.0, 0.9, 0.2, 0.0, 0.1],  # Python tools
    [0.1, 0.8, 0.9, 0.1, 0.0, 0.1, 0.8, 0.2],  # Neural nets
    [0.1, 0.8, 0.7, 0.2, 0.0, 0.1, 0.9, 0.4],  # Transformers
    [0.8, 0.2, 0.1, 0.0, 0.6, 0.4, 0.0, 0.3],  # Python version
])

print(f"  Knowledge base: {len(knowledge_base)} chunks")
print(f"  Each chunk → {embedding_dim}-dim vector\n")

for i, (doc, emb) in enumerate(zip(knowledge_base, embeddings)):
    vec_str = ', '.join(f'{v:.1f}' for v in emb[:4])
    print(f"  {i+1}. '{doc[:45]}{'...' if len(doc) > 45 else ''}'")
    print(f"     → [{vec_str}, ...]")

---
## Step 3: Retrieval — Finding Relevant Chunks

When a user asks a question:
1. Embed the question into a vector
2. Find the K nearest chunks (cosine similarity)
3. Return those chunks as context

In [ ]:
# Vector search (retrieval)
print("Retrieval: Finding Relevant Chunks\n")

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

def retrieve(query_embedding, embeddings, documents, top_k=3):
    similarities = [cosine_similarity(query_embedding, emb) for emb in embeddings]
    ranked = sorted(enumerate(similarities), key=lambda x: -x[1])
    return [(documents[i], sim) for i, sim in ranked[:top_k]]

# Query 1: About Python
query1 = "Who created Python?"
query1_emb = np.array([0.85, 0.1, 0.15, 0.0, 0.75, 0.1, 0.0, 0.1])

print(f"  Query: '{query1}'\n")
results = retrieve(query1_emb, embeddings, knowledge_base, top_k=3)
print(f"  Top 3 results:")
for i, (doc, sim) in enumerate(results, 1):
    bar = '█' * int(sim * 20)
    print(f"    {i}. [{sim:.3f}] {bar} '{doc}'")

# Query 2: About AI
print()
query2 = "How do neural networks work?"
query2_emb = np.array([0.1, 0.85, 0.85, 0.1, 0.0, 0.1, 0.8, 0.2])

print(f"  Query: '{query2}'\n")
results = retrieve(query2_emb, embeddings, knowledge_base, top_k=3)
print(f"  Top 3 results:")
for i, (doc, sim) in enumerate(results, 1):
    bar = '█' * int(sim * 20)
    print(f"    {i}. [{sim:.3f}] {bar} '{doc}'")

print(f"\n  The search finds semantically similar docs, not just keyword matches!")
print(f"  'neural networks work' matches 'biological neurons' by MEANING.")

In [ ]:
# Keyword search vs Semantic search
print("Keyword Search vs Semantic Search:\n")

documents = [
    "The automobile industry is facing challenges.",
    "Cars are becoming increasingly electric.",
    "Vehicle safety standards have improved.",
    "The stock market rose 2% today.",
    "Public transportation reduces traffic.",
]

query = "car problems"

# Keyword search: exact word matching
print(f"  Query: '{query}'\n")
print(f"  Keyword search (contains 'car' or 'problems'):")
for doc in documents:
    if 'car' in doc.lower() or 'problem' in doc.lower():
        print(f"    ✓ '{doc}'")
    else:
        print(f"    ✗ '{doc}'")

print(f"\n  Semantic search (by meaning):")
# Simulate semantic scores
semantic_scores = [0.85, 0.72, 0.68, 0.10, 0.45]
ranked = sorted(zip(documents, semantic_scores), key=lambda x: -x[1])
for doc, score in ranked:
    marker = "✓" if score > 0.5 else "✗"
    print(f"    {marker} [{score:.2f}] '{doc}'")

print(f"\n  Semantic search finds 'automobile...challenges' for 'car problems'")
print(f"  even though no words match! It understands synonyms and concepts.")
print(f"\n  Best practice: combine both (hybrid search)")
print(f"    final_score = 0.7 × semantic_score + 0.3 × keyword_score")

---
## Step 4: Generation — LLM Answers Using Retrieved Context

The retrieved chunks are stuffed into the prompt:

```
System: Answer based ONLY on the provided context.
        If the answer isn't in the context, say "I don't know."

Context:
[Doc 1]: Python was created by Guido van Rossum in 1991.
[Doc 2]: Python uses indentation for code blocks.

User: Who created Python?

Assistant: Python was created by Guido van Rossum in 1991.
```

In [ ]:
# Full RAG prompt construction
print("RAG Prompt Construction:\n")

def build_rag_prompt(query, retrieved_docs):
    context = "\n".join(f"  [Doc {i+1}]: {doc}" for i, (doc, _) in enumerate(retrieved_docs))
    
    prompt = f"""System: Answer the question based ONLY on the provided context.
If the answer is not in the context, say "I don't know."
Cite which document(s) you used.

Context:
{context}

Question: {query}

Answer:"""
    return prompt

# Example 1: answerable question
query1 = "Who created Python and when?"
docs1 = retrieve(
    np.array([0.85, 0.1, 0.15, 0.0, 0.75, 0.1, 0.0, 0.1]),
    embeddings, knowledge_base, top_k=2
)

prompt1 = build_rag_prompt(query1, docs1)
print(f"Example 1 (answerable):\n")
print(f"{prompt1}")
print(f"  Python was created by Guido van Rossum in 1991. [Doc 1]")

# Example 2: unanswerable question
print(f"\n{'─' * 60}\n")
query2 = "What is the population of Tokyo?"
docs2 = retrieve(
    np.array([0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1, 0.1]),  # unrelated
    embeddings, knowledge_base, top_k=2
)

prompt2 = build_rag_prompt(query2, docs2)
print(f"Example 2 (unanswerable):\n")
print(f"{prompt2}")
print(f"  I don't know. The provided context does not contain information about Tokyo's population.")

---
## Vector Databases

For millions of chunks, we need specialized databases that handle vector similarity search efficiently.

```
Regular DB:  SELECT * WHERE name = 'Python'
Vector DB:   SELECT * ORDER BY distance(embedding, query_vector) LIMIT 5
```

Popular vector databases:
- **Pinecone** — fully managed, serverless
- **Weaviate** — open source, hybrid search
- **ChromaDB** — lightweight, Python-native
- **pgvector** — PostgreSQL extension (add vectors to existing DB)
- **FAISS** — Facebook's library for fast similarity search

In [ ]:
# Simple vector database implementation
print("Simple Vector Database (in-memory):\n")

class SimpleVectorDB:
    def __init__(self, embedding_dim):
        self.embeddings = []
        self.documents = []
        self.metadata = []
        self.dim = embedding_dim
    
    def add(self, text, embedding, metadata=None):
        self.documents.append(text)
        self.embeddings.append(np.array(embedding))
        self.metadata.append(metadata or {})
    
    def search(self, query_embedding, top_k=3):
        query = np.array(query_embedding)
        scores = []
        for i, emb in enumerate(self.embeddings):
            sim = np.dot(query, emb) / (np.linalg.norm(query) * np.linalg.norm(emb))
            scores.append((i, sim))
        scores.sort(key=lambda x: -x[1])
        return [(self.documents[i], self.metadata[i], sim) for i, sim in scores[:top_k]]
    
    def __len__(self):
        return len(self.documents)

# Build the database
db = SimpleVectorDB(embedding_dim=8)

# Add documents with metadata
docs_with_meta = [
    ("Python was created in 1991 by Guido van Rossum.", {"source": "wiki", "topic": "python"}),
    ("JavaScript was created in 1995 by Brendan Eich.", {"source": "wiki", "topic": "javascript"}),
    ("Rust was created in 2010 by Graydon Hoare.", {"source": "wiki", "topic": "rust"}),
    ("Go was created in 2009 by Robert Griesemer at Google.", {"source": "wiki", "topic": "go"}),
    ("TypeScript was created in 2012 by Microsoft.", {"source": "blog", "topic": "typescript"}),
]

# Simulated embeddings (similar languages cluster together)
doc_embeddings = [
    [0.9, 0.2, 0.1, 0.8, 0.3, 0.1, 0.1, 0.7],
    [0.2, 0.9, 0.1, 0.7, 0.8, 0.1, 0.2, 0.3],
    [0.3, 0.1, 0.9, 0.2, 0.1, 0.8, 0.7, 0.2],
    [0.4, 0.1, 0.8, 0.3, 0.1, 0.7, 0.6, 0.3],
    [0.2, 0.8, 0.2, 0.6, 0.9, 0.1, 0.2, 0.4],
]

for (text, meta), emb in zip(docs_with_meta, doc_embeddings):
    db.add(text, emb, meta)

print(f"  Database: {len(db)} documents indexed\n")

# Search
queries = [
    ("When was Python created?", [0.85, 0.15, 0.1, 0.75, 0.25, 0.1, 0.1, 0.65]),
    ("What systems languages exist?", [0.3, 0.1, 0.85, 0.2, 0.1, 0.75, 0.65, 0.25]),
]

for query_text, query_emb in queries:
    print(f"  Query: '{query_text}'")
    results = db.search(query_emb, top_k=2)
    for doc, meta, sim in results:
        print(f"    [{sim:.3f}] {doc}  (source: {meta['source']})")
    print()

---
## Improving RAG Quality

Basic RAG has failure modes. Here's how to improve it:

In [ ]:
# RAG failure modes and solutions
print("RAG Failure Modes and Solutions:\n")

failures = [
    {
        "problem": "Retrieved docs are irrelevant",
        "example": "Query: 'Python GIL' → Retrieves: 'Python was created in 1991'",
        "solutions": [
            "Better embeddings (domain-specific models)",
            "Hybrid search (semantic + keyword)",
            "Re-ranking: use a cross-encoder to re-score results",
        ]
    },
    {
        "problem": "Answer spans multiple chunks",
        "example": "Query: 'Compare Python and Rust' → Each chunk only mentions one",
        "solutions": [
            "Larger chunk size (more context per chunk)",
            "Retrieve more chunks (top-10 instead of top-3)",
            "Parent-child chunks (retrieve small, read large)",
        ]
    },
    {
        "problem": "LLM ignores context and hallucinates",
        "example": "Context says X, but LLM generates Y from training data",
        "solutions": [
            "Stronger system prompt ('ONLY use provided context')",
            "Ask model to quote sources",
            "Use a grounded generation model",
        ]
    },
    {
        "problem": "Query is vague or ambiguous",
        "example": "Query: 'How does it work?' (what is 'it'?)",
        "solutions": [
            "Query rewriting (use LLM to clarify the question)",
            "Include conversation history for context",
            "HyDE: generate hypothetical answer, search with that",
        ]
    },
]

for i, f in enumerate(failures, 1):
    print(f"  {i}. PROBLEM: {f['problem']}")
    print(f"     Example: {f['example']}")
    print(f"     Solutions:")
    for s in f['solutions']:
        print(f"       • {s}")
    print()

In [ ]:
# Advanced RAG techniques
print("Advanced RAG Techniques:\n")

print("  1. QUERY REWRITING")
print("     Original: 'How does it compare?'")
print("     Rewritten: 'How does Python compare to JavaScript in performance?'")
print("     (LLM expands the query using conversation context)\n")

print("  2. HyDE (Hypothetical Document Embeddings)")
print("     Query: 'What causes rain?'")
print("     Step 1: LLM generates hypothetical answer:")
print("       'Rain is caused by water evaporating, rising as vapor...'")
print("     Step 2: Embed the hypothetical answer (not the question)")
print("     Step 3: Search with that embedding")
print("     Why: answers are more similar to documents than questions are!\n")

print("  3. RE-RANKING")
print("     Step 1: Retrieve top-20 with fast vector search")
print("     Step 2: Re-score with expensive cross-encoder model")
print("     Step 3: Return top-5 after re-ranking")
print("     (Cross-encoder is more accurate but too slow for full search)\n")

print("  4. PARENT-CHILD CHUNKING")
print("     Index: small chunks (better for precise matching)")
print("     Retrieve: when a small chunk matches, return its PARENT")
print("     (parent = larger surrounding context)")
print("     Example: match on a single sentence, return the full paragraph\n")

print("  5. MULTI-QUERY RAG")
print("     Generate multiple search queries from one question:")
print("     'Compare Python and Rust' → ")
print("       Query A: 'Python advantages and features'")
print("       Query B: 'Rust advantages and features'")
print("       Query C: 'Python vs Rust performance'")
print("     Combine all results → more complete context")

---
## RAG Evaluation

RAG has two components to evaluate:
1. **Retrieval quality**: Did we find the right documents?
2. **Generation quality**: Did the LLM answer correctly given those documents?

In [ ]:
# RAG evaluation metrics
print("RAG Evaluation Metrics:\n")

print("  RETRIEVAL METRICS (did we find the right docs?)\n")

# Simulate retrieval results
# For the query "Who created Python?", these docs are relevant:
relevant_docs = {0, 4}  # indices of relevant documents
retrieved_docs = [0, 2, 4, 5, 1]  # what the system retrieved (ranked)

# Precision@K
k = 3
retrieved_k = retrieved_docs[:k]
relevant_in_k = len(set(retrieved_k) & relevant_docs)
precision_at_k = relevant_in_k / k

print(f"  Query: 'Who created Python?'")
print(f"  Relevant docs: {relevant_docs}")
print(f"  Retrieved (top-{k}): {retrieved_k}\n")

print(f"  Precision@{k} = relevant_in_top_{k} / {k} = {relevant_in_k}/{k} = {precision_at_k:.2f}")

# Recall@K
recall_at_k = relevant_in_k / len(relevant_docs)
print(f"  Recall@{k} = relevant_found / total_relevant = {relevant_in_k}/{len(relevant_docs)} = {recall_at_k:.2f}")

# MRR (Mean Reciprocal Rank)
first_relevant_rank = None
for i, doc_id in enumerate(retrieved_docs):
    if doc_id in relevant_docs:
        first_relevant_rank = i + 1
        break
mrr = 1.0 / first_relevant_rank if first_relevant_rank else 0
print(f"  MRR = 1/rank_of_first_relevant = 1/{first_relevant_rank} = {mrr:.2f}")

print(f"\n\n  GENERATION METRICS (did the LLM answer correctly?)\n")
print(f"  • Faithfulness: Does the answer only use info from context?")
print(f"    (No hallucination — every claim traceable to a doc)")
print(f"  • Relevance: Does the answer address the question?")
print(f"  • Completeness: Does it cover all relevant info from context?")
print(f"\n  Tools: RAGAS framework, TruLens, LLM-as-judge (Ch 17)")

---
## RAG in Production: Architecture

```
┌──────────────────────────────────────────────────────┐
│                    RAG System                         │
│                                                      │
│  ┌─────────┐     ┌──────────┐     ┌─────────────┐  │
│  │ Ingestion│     │ Vector DB│     │    LLM      │  │
│  │ Pipeline │────→│ (FAISS/  │────→│ (Claude/    │  │
│  │          │     │ Pinecone)│     │  GPT-4)     │  │
│  └─────────┘     └──────────┘     └─────────────┘  │
│       ↑                ↑                  ↑         │
│  Documents         Query              Retrieved      │
│  (PDF, web,        embedding          docs +         │
│   Slack, DB)                          user query     │
└──────────────────────────────────────────────────────┘
```

In [ ]:
# Production RAG considerations
print("RAG in Production — Key Considerations:\n")

print("  ┌────────────────────────────────────────────────────────────┐")
print("  │ Component          │ Choices                               │")
print("  ├────────────────────┼────────────────────────────────────────┤")
print("  │ Embedding model    │ OpenAI ada-002, Cohere, E5, BGE      │")
print("  │ Vector DB          │ Pinecone, Weaviate, ChromaDB, FAISS  │")
print("  │ Chunking           │ 200-1000 tokens, with overlap        │")
print("  │ Retrieval          │ Top-5 to Top-20, with re-ranking     │")
print("  │ LLM                │ Claude, GPT-4, Llama (for generation)│")
print("  │ Context window     │ 4K-200K tokens (fits retrieved docs) │")
print("  └────────────────────┴────────────────────────────────────────┘")

print(f"\n  Scaling considerations:")
print(f"    • 1K documents:   FAISS in-memory (free, fast)")
print(f"    • 100K documents: Pinecone/Weaviate (managed, scalable)")
print(f"    • 10M documents:  Distributed vector DB + approximate search")

print(f"\n  Cost breakdown (per query):")
print(f"    Embedding query:    ~$0.0001")
print(f"    Vector search:      ~$0.0001")
print(f"    LLM generation:     ~$0.01-0.05 (dominates cost!)")
print(f"    Total:              ~$0.01-0.05 per query")

print(f"\n  Common frameworks:")
print(f"    • LangChain — popular, many integrations")
print(f"    • LlamaIndex — focused on RAG/data ingestion")
print(f"    • Haystack — open source, production-ready")

---
## Summary

| Concept | Key Point |
|---------|----------|
| **RAG** | Retrieve relevant docs, stuff into prompt, LLM answers |
| **Chunking** | Split docs into searchable pieces (with overlap) |
| **Embeddings** | Convert text to vectors for semantic search |
| **Vector DB** | Specialized database for similarity search |
| **Semantic search** | Find by meaning, not just keywords |
| **Hybrid search** | Combine semantic + keyword for best results |
| **Advanced RAG** | Query rewriting, HyDE, re-ranking, multi-query |
| **Evaluation** | Measure retrieval (precision/recall) + generation (faithfulness) |

**Next chapter**: Deployment — putting models into production